In [3]:
import pandas as pd
import datetime
import os

print(os.getcwd())
filepath = 'Extra\\Leave reports\\Melbourne Dental School Student Leave Form_February 24, 2026_10.28.xlsx'
# Load the file again, skipping the second header row for better parsing (actually row 0 is the data labels)
df = pd.read_excel(filepath, skiprows=[1])

# Select and rename relevant columns
cols_to_keep = {
    'Student ID': 'Student ID',
    'Q3': 'First Name',
    'Q4': 'Last Name',
    'Q6': 'Category',
    'Q7': 'Reason',
    'Q8': 'Start Date',
    'Q9': 'End Date',
    'Q10': 'Cohort',
    'Q30': 'Activities Missed',
}

# Add subject columns (Q4 to Q29)
subject_cols = ['Q4.1', 'Q13', 'Q12', 'Q22', 'Q21', 'Q20', 'Q19', 'Q18', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29']

# Create a clean dataframe
clean_df = df[list(cols_to_keep.keys()) + subject_cols].copy()
clean_df.rename(columns=cols_to_keep, inplace=True)

# Combine subjects into one column
def combine_subjects(row):
    subjects = [str(row[col]) for col in subject_cols if pd.notnull(row[col]) and str(row[col]).lower() != 'nan']
    return ", ".join(subjects)

clean_df['Subjects Missed'] = clean_df.apply(combine_subjects, axis=1)
clean_df.drop(columns=subject_cols, inplace=True)

# Parse dates and calculate duration
def calculate_days(row):
    try:
        start = pd.to_datetime(row['Start Date'], dayfirst=True)
        end = pd.to_datetime(row['End Date'], dayfirst=True)
        return (end - start).days + 1
    except:
        return 0

clean_df['Total Days'] = clean_df.apply(calculate_days, axis=1)

# Format for output
report_df = clean_df[['First Name', 'Last Name', 'Start Date', 'End Date', 'Total Days', 'Cohort', 'Subjects Missed', 'Reason']]

display(report_df.head())

c:\Users\Kunal Patel\D folder\MDS Work\2026


c:\Users\Kunal Patel\D folder\MDS Work\MDS\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,First Name,Last Name,Start Date,End Date,Total Days,Cohort,Subjects Missed,Reason
0,YueHeng,Xu,15/01/2026,16/01/2026,2.0,DDS4,DENT90124 Comprehensive Dental Practice,When I was attempting to purchase airplane tic...
1,test,test,test,test,0.0,DDS3,DENT90150 Dental Practice 3,test
2,NaN,NaN,NaN,NaN,NaN,NaN,,NaN
3,fasf,fasdf,01/02/2026,01/02/2026,1.0,BOH1,"ORAL10001 Society and Health,ORAL10003 Oral He...",asdf
4,Yong,Wang,19/01/2026,23/01/2026,5.0,DDS2,DENT90146 Dental Practice 2,I am writing to formally request approved leav...


In [2]:
# Define a mapping for important columns
important_cols_map = {
    'Student ID': 'Student_ID',
    'Q3': 'First_Name',
    'Q4': 'Last_Name',
    'Q5': 'Email',
    'Q6': 'Leave_Category',
    'Q6_6_TEXT': 'Leave_Category_Other',
    'Q7': 'Detailed_Reason',
    'Q8': 'Start_Date',
    'Q9': 'End_Date',
    'Q10': 'Cohort',
    'Q30': 'Activities_Missed',
    # 'Q32': 'Clinic_Absent',
    # 'Q33': 'Clinic_Days_Missed',
    'Q35': 'Staff_Contacted',
    'Q37_Name': 'Support_Document'
}
# combine leave category and other category into one column
def combine_leave_category(row):
    if pd.notnull(row['Leave_Category_Other']) and str(row['Leave_Category_Other']).lower() != 'nan':
        return f"{row['Leave_Category']} - {row['Leave_Category_Other']}"
    return row['Leave_Category']

# Define subject columns
subject_cols = ['Q4.1', 'Q13', 'Q12', 'Q22', 'Q21', 'Q20', 'Q19', 'Q18', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29']

# Process the data
def get_subjects(row):
    subs = [str(row[c]) for c in subject_cols if pd.notnull(row[c]) and str(row[c]).lower() != 'nan']
    return ", ".join(subs)


# Helper for days calculation
def calc_days(row):
    try:
        start = pd.to_datetime(row['Q8'], dayfirst=True, errors='coerce')
        end = pd.to_datetime(row['Q9'], dayfirst=True, errors='coerce')
        if pd.isnull(start) or pd.isnull(end):
            return 0
        days = (end - start).days + 1
        return days if days > 0 else 0
    except:
        return 0

df['Total_Days'] = df.apply(calc_days, axis=1)

# Keep relevant columns
cols_to_extract = list(important_cols_map.keys()) + ['Subjects_Missed', 'Total_Days']
df['Subjects_Missed'] = df.apply(get_subjects, axis=1)

data = df[cols_to_extract].copy()
data.rename(columns=important_cols_map, inplace=True, errors='ignore')
data['Leave_Category'] = data.apply(combine_leave_category, axis=1)
data.drop(columns=['Leave_Category_Other'], inplace=True, errors='ignore')
# Clean up Clinic Days Missed - convert to numeric
# data['Clinic_Days_Missed'] = pd.to_numeric(data['Clinic_Days_Missed'], errors='coerce').fillna(0)

# Grouping by Student ID
def join_unique(series):
    unique_vals = [str(x) for x in series.dropna().unique() if str(x).lower() != 'nan']
    return " | ".join(unique_vals)

agg_funcs = {
    'First_Name': 'first',
    'Last_Name': 'last',
    'Email': 'first',
    'Cohort': 'first',
    'Leave_Category': join_unique,
    'Detailed_Reason': join_unique,
    'Start_Date': join_unique,
    'End_Date': join_unique,
    'Total_Days': 'sum',
    'Subjects_Missed': lambda x: ", ".join(sorted(list(set(", ".join(x).split(", "))))).strip(", "),
    'Activities_Missed': join_unique,
    'Staff_Contacted': join_unique,
    'Support_Document': join_unique
}

student_report = data.groupby('Student_ID').agg(agg_funcs).reset_index()

# Final cleanup of the subjects string
student_report['Subjects_Missed'] = student_report['Subjects_Missed'].apply(lambda x: x.strip(", ").replace(", , ", ", "))

# Save to CSV
# student_report.to_csv('Aggregated_Student_Leave_Report.csv', index=False)

# Display head for verification
display(student_report.head())
student_report.to_excel('Extra/Leave reports/Aggregated_Student_Leave_Report.xlsx', index=False)
print(f"Total students processed: {len(student_report)}")

,Student_ID,First_Name,Last_Name,Email,Cohort,Leave_Category,Detailed_Reason,Start_Date,End_Date,Total_Days,Subjects_Missed,Activities_Missed,Staff_Contacted,Support_Document
0,1,c,p,q,BOH1,Illness or injury,q,q,q,0,ORAL10005 Oral Health Practice 1,,,
1,100XXXX,Mary,Cherucury,mary.cherucury@unimelb.edu.au,BOH1,Illness or injury,TESTING,30/01/2025,30/01/2026,366,,,,
2,1027970,Jason,Wu,chaksanw@student.unimelb.edu.au,DDS4,Illness or injury,My 48 is mesially angulated and starting to hu...,09/02/2026,09/02/2026,1,DENT90124 Comprehensive Dental Practice,"DENT90124: clinic, operator",Yes,
3,1079984,Gabrielle,Mahon,gmahon@student.unimelb.edu.au,DDS4,Illness or injury,Woke up feeling unwell - thus was not able to ...,5/2/2026,5/2/2026,1,DENT90124 Comprehensive Dental Practice,DENT90124 Thursday: DTC Clinic + DTC SND,Yes,
4,1082018,Zhanxu,Li,Zhanxul@student.unimelb.edu.au,DDS4,Illness or injury,"Spine injury due to exercise, unable to move/s...",10/02/2026,10/02/2026,1,DENT90124 Comprehensive Dental Practice,DENT90124 Tuesday: Oral med observation (9-12)...,Yes,


Total students processed: 89


In [6]:
import pandas as pd

# Load previous data (Sheet 0 from the Feb 24 file)
prev_df = pd.read_excel(filepath, skiprows=[1])

# Load new data (Sheet 2 from the 1-1777 file)
new_df = pd.read_excel('Extra/Leave reports/Melbourne Dental School Student Leave Form(1-1777).xlsx', sheet_name='Sheet2')

# --- MAPPING PREVIOUS DATA (prev_df) ---
prev_map = {
    'Student ID': 'Student_ID',
    'Q3': 'First_Name',
    'Q4': 'Last_Name',
    'Q5': 'Email',
    'Q6': 'Leave_Category',
    'Q6_6_TEXT': 'Leave_Category_Other',
    'Q39': 'Planned_Leave_Days',
    'Q7': 'Detailed_Reason',
    'Q8': 'Start_Date',
    'Q9': 'End_Date',
    'Q10': 'Cohort',
    'Q30': 'Activities_Missed',
    'Q35': 'Staff_Contacted',
    'Q37_Name': 'Support_Document'
}
prev_sub_cols = ['Q4.1', 'Q13', 'Q12', 'Q22', 'Q21', 'Q20', 'Q19', 'Q18', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29']

# --- MAPPING NEW DATA (new_df) ---
new_map = {
    'Student ID': 'Student_ID',
    'First Name': 'First_Name',
    'Family Name': 'Last_Name',
    'Email': 'Email',
    'Please select the category of your leave request': 'Leave_Category',
    'Reason for leave': 'Detailed_Reason',
    'Absence Start Date': 'Start_Date',
    'Absence End Date': 'End_Date',
    'Please select your cohort': 'Cohort',
    "List the activities you'll miss during your leave. Please include your allocated role for any placements (i.e Operator).": 'Activities_Missed',
    'Who have you contacted to advise of your absence? Please include the contact method (i.e email)': 'Staff_Contacted',
    'Upload supporting documentation (such a medical certificate)': 'Support_Document'
}
new_sub_cols = [
    'Please select all BOH1 subjects you will be missing', 'Please select all BOH2 subjects you will be missing',
    'Please select all BOH3 subjects you will be missing', 'Please select all the DDS1 subjects you will be missing',
    'Please select all the DDS2 subjects you will be missing', 'Please select all the DDS3 subjects you will be missing',
    'Please select all the DCD Endodontics subjects you will be missing', 'Please select all the DCD Oral Medicine subjects you will be missing',
    'Please select all the DCD Orthodontics subjects you will be missing', 'Please select all the DCD Paediatric Dentistry subjects you will be missing',
    'Please select all the DCD Periodontics subjects you will be missing', 'Please select all the DCD Prosthodontics subjects you will be missing',
    'Please select all the DCD Special Needs Dentistry subjects you will be missing'
]

# --- PROCESSING PREVIOUS DATA ---
def combine_leave_category_prev(row):
    if pd.notnull(row['Q6_6_TEXT']) and str(row['Q6_6_TEXT']).lower() != 'nan':
        return f"{row['Q6']} - {row['Q6_6_TEXT']}"
    return row['Q6']

def get_subjects_prev(row):
    subs = [str(row[c]) for c in prev_sub_cols if c in row and pd.notnull(row[c]) and str(row[c]).lower() != 'nan']
    return ", ".join(subs)

def calc_days_prev(row):
    try:
        start = pd.to_datetime(row['Q8'], dayfirst=True, errors='coerce')
        end = pd.to_datetime(row['Q9'], dayfirst=True, errors='coerce')
        if pd.isnull(start) or pd.isnull(end): return 0
        days = (end - start).days + 1
        return days if days > 0 else 0
    except: return 0



prev_df['Subjects_Missed'] = prev_df.apply(get_subjects_prev, axis=1)
prev_df['Total_Days'] = prev_df.apply(calc_days_prev, axis=1)
prev_df['Leave_Category_Combined'] = prev_df.apply(combine_leave_category_prev, axis=1)

# combine all three Q39 'Q39_4_TEXT'	'Q39_6_TEXT' into one column if not null
def cleanText(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def combineLeave(row):
    q39 = cleanText(row.get("Q39"))
    q39Hours = cleanText(row.get("Q39_4_TEXT"))
    q39Other = cleanText(row.get("Q39_6_TEXT"))

    if q39.lower().startswith("hours"):
        return f'Hours: {q39Hours}' if q39Hours else q39
    if q39.lower() == "other":
        return f'Other: {q39Other}' if q39Other else q39
    return q39
prev_df['Q39'] = prev_df.apply(combineLeave, axis=1)
clean_prev = prev_df[list(prev_map.keys()) + ['Subjects_Missed', 'Total_Days', 'Leave_Category_Combined']].copy()
# Map prev columns to standardized names
clean_prev.rename(columns=prev_map, inplace=True)
clean_prev['Leave_Category'] = clean_prev['Leave_Category_Combined'] # Use the combined one
clean_prev.drop(columns=['Leave_Category_Other', 'Leave_Category_Combined'], inplace=True, errors='ignore')

# --- PROCESSING NEW DATA ---
def get_subjects_new(row):
    subs = [str(row[c]) for c in new_sub_cols if c in row and pd.notnull(row[c]) and str(row[c]).lower() != 'nan']
    return ", ".join(subs)

def calc_days_new(row):
    try:
        start = pd.to_datetime(row['Absence Start Date'], dayfirst=True, errors='coerce')
        end = pd.to_datetime(row['Absence End Date'], dayfirst=True, errors='coerce')
        if pd.isnull(start) or pd.isnull(end): return 0
        days = (end - start).days + 1
        return days if days > 0 else 0
    except: return 0

new_df['Subjects_Missed'] = new_df.apply(get_subjects_new, axis=1)
new_df['Total_Days'] = new_df.apply(calc_days_new, axis=1)
# display(new_df)

clean_new = new_df[list(new_map.keys()) + ['Subjects_Missed', 'Total_Days']].copy()
clean_new.rename(columns=new_map, inplace=True)

# --- COMBINE AND AGGREGATE ---
# combined_all = pd.concat([clean_prev, clean_new], ignore_index=True)
combined_all = clean_prev.copy()
combined_all['Student_ID'] = combined_all['Student_ID'].astype(str).str.strip()
combined_all.drop('Total_Days', axis=1, inplace=True, errors='ignore') 
display(combined_all.head())

def join_unique(series):
    unique_vals = [str(x) for x in series.dropna().unique() if str(x).lower() != 'nan' and str(x).strip() != '']
    return " | ".join(unique_vals)

agg_funcs = {
    'First_Name': 'first',
    'Last_Name': 'last',
    'Email': 'first',
    'Cohort': 'first',
    'Leave_Category': join_unique,
    'Detailed_Reason': join_unique,
    'Start_Date': join_unique,
    'End_Date': join_unique,
    'Planned_Leave_Days': join_unique,
    # 'Total_Days': 'sum',
    'Subjects_Missed': lambda x: ", ".join(sorted(list(set(", ".join(x.astype(str)).split(", "))))).strip(", "),
    'Activities_Missed': join_unique,
    'Staff_Contacted': join_unique,
    'Support_Document': join_unique
}

final_report = combined_all.groupby('Student_ID').agg(agg_funcs).reset_index()

# Final cleanup of strings
final_report['Subjects_Missed'] = final_report['Subjects_Missed'].apply(lambda x: x.strip(", ").replace(", , ", ", "))

# Save to Excel as requested
final_report.to_excel('Extra/Leave reports/Standardized_Combined_Leave_Report.xlsx', index=False)

print(f"Total entries combined: {len(combined_all)}")
print(f"Total unique students: {len(final_report)}")
display(final_report[['Student_ID', 'First_Name', 'Last_Name', 'Planned_Leave_Days', 'Subjects_Missed']].head())

c:\Users\Kunal Patel\D folder\MDS Work\MDS\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Student_ID,First_Name,Last_Name,Email,Leave_Category,Planned_Leave_Days,Detailed_Reason,Start_Date,End_Date,Cohort,Activities_Missed,Staff_Contacted,Support_Document,Subjects_Missed
0,1213093,YueHeng,Xu,yhxu@student.unimelb.edu.au,Other - Physically unable to attend due to unf...,2 or more days,When I was attempting to purchase airplane tic...,15/01/2026,16/01/2026,DDS4,DENT90124 Thursday: Orientation Day 1\nDENT901...,No,support document1.pdf,DENT90124 Comprehensive Dental Practice
1,12345,test,test,elice.chen@unimelb.edu.au,Illness or injury,Half day,test,test,test,DDS3,NaN,NaN,NaN,DENT90150 Dental Practice 3
2,nan,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
3,123,fasf,fasdf,fasfd,Illness or injury,Half day,asdf,01/02/2026,01/02/2026,BOH1,NaN,Yes,NaN,"ORAL10001 Society and Health,ORAL10003 Oral He..."
4,1361370,Yong,Wang,yowang2@student.unimelb.edu.au,Religious obligations,2 or more days,I am writing to formally request approved leav...,19/01/2026,23/01/2026,DDS2,"Orientation Monday, OHV Titanium Training Tues...",No,NaN,DENT90146 Dental Practice 2


Total entries combined: 113
Total unique students: 89


,Student_ID,First_Name,Last_Name,Planned_Leave_Days,Subjects_Missed
0,1,c,p,1 day,ORAL10005 Oral Health Practice 1
1,100XXXX,Mary,Cherucury,1 day,
2,1027970,Jason,Wu,1 day,DENT90124 Comprehensive Dental Practice
3,1079984,Gabrielle,Mahon,1 day,DENT90124 Comprehensive Dental Practice
4,1082018,Zhanxu,Li,1 day,DENT90124 Comprehensive Dental Practice


In [7]:
from Utils import getmodeArgs
# separate combined_all based on cohort
cohorts = combined_all['Cohort'].dropna().unique()
filename = f'Extra/Leave reports/Combined_Leave_Report.xlsx'
combined_all.to_excel(filename, index=False, sheet_name='All')  # Save full combined report first
for cohort in cohorts:
    cohort_df = combined_all[combined_all['Cohort'] == cohort]
    wargs = getmodeArgs(filename)
    with pd.ExcelWriter(filename, **wargs) as writer:
        cohort_df.to_excel(writer, sheet_name=cohort[:31], index=False)



(841.68, 1190.8799999999999)


In [ ]:
cohorts = final_report['Cohort'].dropna().unique()
cols = ['Student_ID', 'First_Name', 'Last_Name', 'Cohort', 'Start_Date', 'End_Date', 'Planned_Leave_Days', 'Subjects_Missed', 'Activities_Missed']
writer_args  = {}

for cohort in cohorts:
    cohort_df = final_report[final_report['Cohort'] == cohort]
    print(f"Cohort: {cohort} - Total Students: {len(cohort_df)}")
    # different sheets for each cohort
    filename = f'Extra/Leave reports/Leave_Report_By_Cohort.xlsx'
    if os.path.exists(filename):
        writer_args['mode'] = 'a'
        writer_args['engine'] = 'openpyxl'
        writer_args['if_sheet_exists'] = 'replace'
    else:
        writer_args['engine'] = 'openpyxl'
        writer_args['mode'] = 'w'
    with pd.ExcelWriter(filename, **writer_args) as writer:
        cohort_df[cols].to_excel(writer, index=False, sheet_name=cohort)